## Model Confidence

We wish to investigate the correctness of the preduciton based on the value returned by the model. <br>
That is, when the model returns a value closer to one, <br>
is it indeed more likely that the tissue is cancerous, <br>
and vice versa, if the value close to zero, <br>
is there a high probability that the tissue is healthy?

In [ ]:
import sys
from google.colab import drive
drive.mount("/content/drive")
PROJECT_ROOT = "/content/drive/MyDrive/Projects/MMSEN"
sys.path.insert(0, PROJECT_ROOT)

In [ ]:
!kaggle datasets download -d andrewmvd/metastatic-tissue-classification-patchcamelyon
!unzip metastatic-tissue-classification-patchcamelyon.zip

In [ ]:
import torch
from tqdm import tqdm
from matplotlib import pyplot as plt

from src.data import load_data
from src.utils import assemble_mmsen_small

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [ ]:
# load the final model
mmsen_final = assemble_mmsen_small()
mmsen_final_pth = torch.load('/content/drive/MyDrive/mmsen_final.pth', weights_only=False, map_location=torch.device(device))
mmsen_final.load_state_dict(mmsen_final_pth['model_state_dict'])
mmsen_final = mmsen_final.to(device, memory_format=torch.channels_last)

In [ ]:
bins = [[0 , 0] for _ in range(10)]
_, _, val_loader = load_data()

for batch in tqdm(val_loader, total=len(val_loader)):
  x = batch[0].to(device)
  y = batch[1].to(device)

  with torch.inference_mode():

    pred = torch.sigmoid(mmsen_final(x))
    pred_label = torch.round(pred)

  for i in range(10):

    mask = (pred >= i*0.1) & (pred < (i+1)*0.1)
    passed = pred[mask]
    bins[i][0] += passed[pred_label[mask].squeeze() != y[mask].squeeze()].numel()
    bins[i][1] += passed[pred_label[mask].squeeze() == y[mask].squeeze()].numel()

  bins

In [ ]:
bins_flattened = [val for bin in bins for val in bin]

In [ ]:
x = [i + 1*(i//2) for i in range(20)]
colors = 10*['red', 'green']
tick_label = [f'>{i}%' if i%2==0 else ' ' for i in range(-5, 91, 5)]

The green bar shows the number of correct predictions,
the red bar the wrong predicitons for each partition.

In [1]:
plt.bar(x, bins_flattened, color=colors, tick_label=tick_label)
plt.title('Partitioning correct detections by Likelyhood')

NameError: name 'plt' is not defined

In [ ]:
print('       False    |    Correct')
print('-----------------------------')
for i, (row, bin) in enumerate(zip([f'>{i}%' for i in range(0, 91, 10)], bins)):
  if i == 5:
    print('-----------------------------')
  if i == 0:
    print(f'{row}      {bin[0]}    |   {bin[1]}')
  else:
    print(f'{row}     {bin[0]}    |   {bin[1]}')

For samples classified as cancerous, false positives are constantly decreasig while true positives are constarntly increasing, with increasing confidence.

In [ ]:
plt.bar([f'>{i}%' for i in range(0, 91, 10)], [bin[1]/sum(bin) for bin in bins], color='green')
plt.title('Correct predictions relative to total predictions')

As the model becomes more confident in its prediction the results also become more reliable.